In [59]:
import os
import pandas as pd

In [60]:
# Go up one level from /notebooks to project root, then into /data
file_path = os.path.join("..", "data", "df_final.csv")

df = pd.read_csv(file_path)
print(df.head())

   age_bin  sex  cp  trestbps_scaled  chol_scaled  fbs  restecg  \
0        5    1   1         0.750380    -0.191089    1        2   
1        5    1   4         1.596266     0.815023    0        2   
2        5    1   4        -0.659431    -0.276057    0        2   
3        2    1   3        -0.095506     0.154523    0        0   
4        3    0   2        -0.095506    -0.843122    0        2   

   thalach_scaled  exang  oldpeak  slope   ca  thal  target  target_binary  
0        0.093815      0      2.3      3  0.0   6.0       0              0  
1       -1.898316      1      1.5      2  3.0   3.0       2              1  
2       -0.821444      1      2.6      2  2.0   7.0       1              1  
3        1.433358      0      3.5      3  0.0   3.0       0              0  
4        0.925134      0      1.4      1  0.0   3.0       0              0  


In [61]:
print(f"Dataset size: {df.shape}")
print(df['target_binary'].value_counts(normalize=True))

Dataset size: (297, 15)
target_binary
0    0.538721
1    0.461279
Name: proportion, dtype: float64


### The distribution of target label values are nearly similar, however, the dataset is quite small. Hence while splitting, stratify will be applied.

In [62]:
from sklearn.model_selection import train_test_split

y = df["target_binary"]
X = df.drop("target_binary", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # <-- ensures balanced split
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


target_binary
0    0.540084
1    0.459916
Name: proportion, dtype: float64
target_binary
0    0.533333
1    0.466667
Name: proportion, dtype: float64


### Distribution of train and test are maintained in similar proportion

In [63]:
import sys, os
sys.path.append(os.path.abspath("../src"))

from mlflow_config import setup_mlflow

# Setup mlflow experiment
experiment_name = "heart-disease-model-selection"
setup_mlflow(experiment_name)


MLflow tracking set to: d:\Bits-lab\mlops-heart-classifier\mlflow_tracking
Experiment: heart-disease-model-selection


In [64]:
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import itertools

In [65]:
# Logistic Regression
# Define parameter grid
param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "saga"],
    "l1_ratio": [0]   # equivalent to penalty="l2"
}

# Generate all combinations
param_combinations = list(itertools.product(
    param_grid_lr["C"],
    param_grid_lr["solver"],
    param_grid_lr["l1_ratio"]
))

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for C, solver, l1_ratio in param_combinations:
    with mlflow.start_run(run_name=f"LogReg_C{C}_solver{solver}_l1ratio{l1_ratio}"):
        # Define model with current params
        log_reg = LogisticRegression(
            C=C,
            solver=solver,
            l1_ratio=l1_ratio,   # <-- use l1_ratio instead of penalty
            max_iter=1000,
            random_state=42
        )

        # Cross-validation ROC-AUC
        cv_scores = cross_val_score(log_reg, X_train, y_train, cv=cv, scoring="roc_auc")
        mean_cv_score = cv_scores.mean()

        # Log parameters and CV metric
        mlflow.log_params({
            "model_name": "LogisticRegression",
            "C": C,
            "solver": solver,
            "l1_ratio": l1_ratio
        })
        mlflow.log_metric("cv_roc_auc", float(mean_cv_score))

        log_reg.fit(X_train, y_train)
        y_pred = log_reg.predict(X_test)
        y_proba = log_reg.predict_proba(X_test)[:, 1]

        # Log test metrics
        mlflow.log_metric("test_accuracy", float(accuracy_score(y_test, y_pred)))
        mlflow.log_metric("test_precision", float(precision_score(y_test, y_pred)))
        mlflow.log_metric("test_recall", float(recall_score(y_test, y_pred)))
        mlflow.log_metric("test_f1", float(f1_score(y_test, y_pred)))
        mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, y_proba)))


In [66]:

def retrieve_best_run(experiment_name, model_name):
    # Retrieve the best hyper parameter set from hte experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)

    # Filter runs by model_name tag or param
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"params.model_name = '{model_name}'",   # or params.model_name
        #filter_string = "",
        run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
        output_format="pandas"
    )

    # Ensure start_time and end_time are datetime objects
    runs_df["start_time"] = pd.to_datetime(runs_df["start_time"])
    runs_df["end_time"]   = pd.to_datetime(runs_df["end_time"])

    # Calculate duration in seconds
    runs_df["duration_sec"] = (runs_df["end_time"] - runs_df["start_time"]).dt.total_seconds()

    # Sort by best ROC-AUC and shortest duration
    best_run = runs_df.sort_values(
        by=["metrics.cv_roc_auc", "duration_sec"],
        ascending=[False, True]
    ).iloc[0]

    return best_run


In [67]:
best_run_lg = retrieve_best_run(experiment_name, "LogisticRegression")
print(best_run_lg)

run_id                                      c505f3b2765a40ac9a98c0d20f5dfce9
experiment_id                                                              1
status                                                              FINISHED
artifact_uri               file:///d:/Bits-lab/mlops-heart-classifier/mlf...
start_time                                  2026-10-19 15:45:52.900000+00:00
end_time                                    2026-10-19 15:45:53.269000+00:00
metrics.cv_roc_auc                                                       1.0
metrics.test_f1                                                          NaN
metrics.test_accuracy                                                    NaN
metrics.test_roc_auc                                                     NaN
metrics.test_precision                                                   NaN
metrics.test_recall                                                      NaN
params.model_name                                         LogisticRegression

In [68]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import itertools

# Define parameter grid manually
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", "log2"]
}

# Generate all combinations
param_combinations = list(itertools.product(
    param_grid_rf["n_estimators"],
    param_grid_rf["max_depth"],
    param_grid_rf["max_features"]
))

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for n_estimators, max_depth, max_features in param_combinations:
    with mlflow.start_run(run_name=f"RF_n{n_estimators}_d{max_depth}_f{max_features}"):
        # Define model with current params
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_features=max_features,
            random_state=42
        )

        # Cross-validation ROC-AUC
        cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="roc_auc")
        mean_cv_score = cv_scores.mean()

        # Log parameters and CV metric
        mlflow.log_params({
            "model_name": "RandomForest",
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "max_features": max_features
        })
        mlflow.log_metric("cv_roc_auc", float(mean_cv_score))

        # Fit on full training set
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        y_proba = rf.predict_proba(X_test)[:, 1]

        # Log test metrics
        mlflow.log_metric("test_accuracy", float(accuracy_score(y_test, y_pred)))
        mlflow.log_metric("test_precision", float(precision_score(y_test, y_pred)))
        mlflow.log_metric("test_recall", float(recall_score(y_test, y_pred)))
        mlflow.log_metric("test_f1", float(f1_score(y_test, y_pred)))
        mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, y_proba)))

        # Log model artifact
        mlflow.sklearn.log_model(
                                    rf,
                                    name="rf_model",
                                    skops_trusted_types=["sklearn.tree._tree.Tree"]
        )


In [69]:
best_run_rf = retrieve_best_run(experiment_name, "RandomForest")
print(best_run_rf)

run_id                                      bdcd82468c72450ab688ecf9cf3ed7d5
experiment_id                                                              1
status                                                              FINISHED
artifact_uri               file:///d:/Bits-lab/mlops-heart-classifier/mlf...
start_time                                  2026-10-19 16:24:43.457000+00:00
end_time                                    2026-10-19 16:24:50.312000+00:00
metrics.cv_roc_auc                                                       1.0
metrics.test_f1                                                          1.0
metrics.test_accuracy                                                    1.0
metrics.test_roc_auc                                                     1.0
metrics.test_precision                                                   1.0
metrics.test_recall                                                      1.0
params.model_name                                               RandomForest

In [86]:
import time
# Logistic Regression
from sklearn.model_selection import cross_validate
from sklearn.metrics import confusion_matrix
# Define model
start_time = time.time()
log_reg = LogisticRegression(
            C=int(best_run_lg["params.C"]),
            solver=best_run_lg["params.solver"],
            l1_ratio=int(best_run_lg["params.l1_ratio"]),   # <-- use l1_ratio instead of penalty
            max_iter=1000,
            random_state=42
        )

# Step 1: Cross-validation with multiple metrics
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_results = cross_validate(log_reg, X_train, y_train, cv=5, scoring=scoring)

# Print mean CV scores
print("Logistic Regression CV Results:")
for metric in scoring:
    mean_score = cv_results[f"test_{metric}"].mean()
    print(f"  {metric}: {mean_score:.3f}")

# Step 2: Fit final model on full training set
log_reg.fit(X_train, y_train)

# Step 3: Evaluate on held-out test set
y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:,1]
runtime = time.time() - start_time
print("\nLogistic Regression Test Results:")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  Precision:", precision_score(y_test, y_pred))
print("  Recall:", recall_score(y_test, y_pred))
print("  F1-score:", f1_score(y_test, y_pred))
print("  ROC-AUC:", roc_auc_score(y_test, y_proba))
print("  Runtime:", runtime)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Logistic Regression CV Results:
  accuracy: 1.000
  precision: 1.000
  recall: 1.000
  f1: 1.000
  roc_auc: 1.000

Logistic Regression Test Results:
  Accuracy: 1.0
  Precision: 1.0
  Recall: 1.0
  F1-score: 1.0
  ROC-AUC: 1.0
  Runtime: 0.12899541854858398
Confusion Matrix:
 [[32  0]
 [ 0 28]]


In [87]:
# Run CV for evaluation
from sklearn.ensemble import RandomForestClassifier
start_time = time.time()
rf = RandomForestClassifier(
            n_estimators=int(best_run_rf["params.n_estimators"]),
            max_depth=int(best_run_rf["params.max_depth"]),
            max_features=best_run_rf["params.max_features"],
            random_state=42
        )
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_results = cross_validate(rf, X_train, y_train, cv=5, scoring=scoring)

for metric in scoring:
    print(f"CV {metric}: {cv_results[f'test_{metric}'].mean():.3f}")

# ✅ Fit final model on full training set
rf.fit(X_train, y_train)

# Now you can safely evaluate on the test set
print("Test Accuracy:", rf.score(X_test, y_test))

# Step 3: Evaluate on held-out test set
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:,1]
runtime = time.time() - start_time
print(" Random Forest Test Results:")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  Precision:", precision_score(y_test, y_pred))
print("  Recall:", recall_score(y_test, y_pred))
print("  F1-score:", f1_score(y_test, y_pred))
print("  ROC-AUC:", roc_auc_score(y_test, y_proba))
print("  Runtime:", runtime)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


CV accuracy: 1.000
CV precision: 1.000
CV recall: 1.000
CV f1: 1.000
CV roc_auc: 1.000
Test Accuracy: 1.0
 Random Forest Test Results:
  Accuracy: 1.0
  Precision: 1.0
  Recall: 1.0
  F1-score: 1.0
  ROC-AUC: 1.0
  Runtime: 1.0424575805664062
Confusion Matrix:
 [[32  0]
 [ 0 28]]


In [88]:
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(rf, "../models/rf_model.pkl")
joblib.dump(log_reg, "../models/lg_model.pkl")

['../models/lg_model.pkl']